In [ ]:
import pandas as pd

# 데이터 불러오기
df = pd.read_csv("폭염_위험도_점수표.csv", encoding="utf-8-sig")

# Min-Max 정규화를 적용할 컬럼
normalize_cols = ["방문자수", "수급권자수", "문화행사수"]

# Min-Max 정규화
for col in normalize_cols:
    df[f"{col}_정규화"] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

# 결과 확인
normalized_result = df[
    [
        "자치구",
        "방문자수",
        "수급권자수",
        "문화행사수",
        "방문자수_정규화",
        "수급권자수_정규화",
        "문화행사수_정규화",
    ]
].round(4)

normalized_result

,자치구,방문자수,수급권자수,문화행사수,방문자수_정규화,수급권자수_정규화,문화행사수_정규화
0,강북구,1588445,9906,94,0.3374,0.7744,0.0582
1,강서구,1486544,11680,38,0.3157,0.9513,0.0000
2,노원구,1500160,12169,138,0.3186,1.0000,0.1040
3,도봉구,2358390,9014,51,0.5009,0.6855,0.0135
4,중랑구,820226,10865,55,0.1742,0.8700,0.0177
5,양천구,2717348,8562,59,0.5771,0.6404,0.0218
6,마포구,3071852,4947,227,0.6524,0.2800,0.1965
7,은평구,223979,10067,195,0.0476,0.7904,0.1632
8,강남구,3274011,4208,116,0.6954,0.2064,0.0811
9,광진구,1918843,6138,147,0.4075,0.3988,0.1133


In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("폭염_위험도_점수표.csv", encoding="utf-8-sig")

# -----------------------------
# 1. 수급권자 천 명당 쉼터 수 계산
# -----------------------------
df["수급권자_천명당_쉼터수"] = df["쉼터수"] / (df["수급권자수"] / 1000)

# Min-Max 정규화
min_val = df["수급권자_천명당_쉼터수"].min()
max_val = df["수급권자_천명당_쉼터수"].max()

df["수급권자_천명당_쉼터수_정규화"] = (
    df["수급권자_천명당_쉼터수"] - min_val
) / (max_val - min_val)

# 쉼터가 부족할수록 위험도가 높게 나오도록 역변환
df["쉼터_부족도"] = 1 - df["수급권자_천명당_쉼터수_정규화"]


# -----------------------------
# 2. 기온 및 체감온도 위험도 산출
# 기준: 30℃ 초과 정도
# -----------------------------
df["기온_위험도"] = np.where(
    df["평균기온"] > 30,
    df["평균기온"] - 30,
    0
)

df["체감온도_위험도"] = np.where(
    df["평균체감온도"] > 30,
    df["평균체감온도"] - 30,
    0
)

# 필요 시 0~1 정규화
for col in ["기온_위험도", "체감온도_위험도"]:
    min_val = df[col].min()
    max_val = df[col].max()

    if max_val != min_val:
        df[f"{col}_정규화"] = (df[col] - min_val) / (max_val - min_val)
    else:
        df[f"{col}_정규화"] = 0


# -----------------------------
# 3. 폭염특보 위험도 변환
# 폭염주의보 = 1점, 폭염경보 = 2점 예시
# -----------------------------
warning_df = pd.read_csv("폭염특보 데이터(전처리).csv", encoding="utf-8-sig")

warning_weight = {
    "폭염주의보": 1,
    "폭염경보": 2
}

warning_df["폭염특보_가중치"] = warning_df["특보종류"].map(warning_weight).fillna(0)

폭염특보_위험도 = warning_df["폭염특보_가중치"].sum()

df["폭염특보_위험도"] = 폭염특보_위험도


# -----------------------------
# 4. 인구 1만 명당 쉼터 수 계산
# -----------------------------
# 인구 데이터 불러오기
pop_df = pd.read_csv("서울시_인구밀도_전처리.csv", encoding="utf-8-sig")

# 컬럼명 맞추기
pop_df = pop_df.rename(columns={"동": "자치구", "인구 (명)": "인구수"})

df = df.merge(
    pop_df[["자치구", "인구수"]],
    on="자치구",
    how="left"
)

df["1만명당_쉼터수"] = df["쉼터수"] / (df["인구수"] / 10000)


# -----------------------------
# 5. 쉼터 부족 지역 분류
# 평균보다 낮으면 부족 지역으로 판단
# -----------------------------
shelter_mean = df["1만명당_쉼터수"].mean()

df["쉼터_상태"] = np.where(
    df["1만명당_쉼터수"] < shelter_mean,
    "쉼터 부족",
    "쉼터 양호"
)


# -----------------------------
# 6. 취약지역 분석 지표 생성
# 쉼터 부족도 + 수급권자수 정규화 + 폭염 위험도 반영
# -----------------------------
vulnerable_cols = ["수급권자수", "기온_위험도", "체감온도_위험도"]

for col in vulnerable_cols:
    min_val = df[col].min()
    max_val = df[col].max()

    if max_val != min_val:
        df[f"{col}_정규화"] = (df[col] - min_val) / (max_val - min_val)
    else:
        df[f"{col}_정규화"] = 0

df["취약지역_분석지표"] = (
    df["쉼터_부족도"] * 0.4
    + df["수급권자수_정규화"] * 0.3
    + df["기온_위험도_정규화"] * 0.15
    + df["체감온도_위험도_정규화"] * 0.15
)

# 취약지역 순위
df["취약지역_순위"] = df["취약지역_분석지표"].rank(
    ascending=False,
    method="min"
).astype(int)


# -----------------------------
# 7. 결과 확인
# -----------------------------
result = df[
    [
        "자치구",
        "쉼터수",
        "수급권자수",
        "수급권자_천명당_쉼터수",
        "쉼터_부족도",
        "평균기온",
        "기온_위험도",
        "평균체감온도",
        "체감온도_위험도",
        "폭염특보_위험도",
        "인구수",
        "1만명당_쉼터수",
        "쉼터_상태",
        "취약지역_분석지표",
        "취약지역_순위",
    ]
].round(4)

result

,자치구,쉼터수,수급권자수,수급권자_천명당_쉼터수,쉼터_부족도,평균기온,기온_위험도,평균체감온도,체감온도_위험도,폭염특보_위험도,인구수,1만명당_쉼터수,쉼터_상태,취약지역_분석지표,취약지역_순위
0,강북구,96,9906,9.6911,1.0000,27.03,0.0,31.3,1.3,75,285900,3.3578,쉼터 부족,0.6323,1
1,강서구,189,11680,16.1815,0.8463,27.03,0.0,31.3,1.3,75,556370,3.3970,쉼터 부족,0.6239,2
2,노원구,312,12169,25.6389,0.6223,27.03,0.0,31.3,1.3,75,489003,6.3803,쉼터 양호,0.5489,4
3,도봉구,167,9014,18.5267,0.7907,27.03,0.0,31.3,1.3,75,303051,5.5106,쉼터 양호,0.5219,6
4,중랑구,159,10865,14.6341,0.8829,27.03,0.0,31.3,1.3,75,383764,4.1432,쉼터 부족,0.6142,3
5,양천구,197,8562,23.0086,0.6846,27.03,0.0,31.3,1.3,75,428537,4.5970,쉼터 양호,0.4660,8
6,마포구,85,4947,17.1821,0.8226,27.03,0.0,31.3,1.3,75,369364,2.3013,쉼터 부족,0.4130,12
7,은평구,202,10067,20.0656,0.7543,27.03,0.0,31.3,1.3,75,459586,4.3953,쉼터 부족,0.5389,5
8,강남구,89,4208,21.1502,0.7286,27.03,0.0,31.3,1.3,75,562508,1.5822,쉼터 부족,0.3533,15
9,광진구,115,6138,18.7357,0.7858,27.03,0.0,31.3,1.3,75,349117,3.2940,쉼터 부족,0.4339,11


In [9]:
df["수급권자_천명당_쉼터수"] = df["쉼터수"] / (df["수급권자수"] / 1000)

df["수급권자_천명당_쉼터수_정규화"] = (
    df["수급권자_천명당_쉼터수"] - df["수급권자_천명당_쉼터수"].min()
) / (
    df["수급권자_천명당_쉼터수"].max()
    - df["수급권자_천명당_쉼터수"].min()
)

df["쉼터부족도"] = 1 - df["수급권자_천명당_쉼터수_정규화"]

In [11]:
import pandas as pd

df = pd.read_csv("폭염_위험도_점수표.csv", encoding="utf-8-sig")

# 쉼터부족도 컬럼이 없으면 생성
if "쉼터부족도" not in df.columns:
    df["수급권자_천명당_쉼터수"] = df["쉼터수"] / (df["수급권자수"] / 1000)

    df["수급권자_천명당_쉼터수_정규화"] = (
        df["수급권자_천명당_쉼터수"] - df["수급권자_천명당_쉼터수"].min()
    ) / (
        df["수급권자_천명당_쉼터수"].max()
        - df["수급권자_천명당_쉼터수"].min()
    )

    df["쉼터부족도"] = 1 - df["수급권자_천명당_쉼터수_정규화"]

# 분석에 사용할 지표 정규화
cols = ["방문자수", "수급권자수", "쉼터부족도"]

for col in cols:
    df[f"{col}_정규화"] = (
        df[col] - df[col].min()
    ) / (
        df[col].max() - df[col].min()
    )

# 기존 가중치와 변경 가중치
base_weights = {
    "방문자수_정규화": 0.20,
    "수급권자수_정규화": 0.20,
    "쉼터부족도_정규화": 0.20
}

changed_weights = {
    "방문자수_정규화": 0.15,
    "수급권자수_정규화": 0.25,
    "쉼터부족도_정규화": 0.20
}

# 위험도 점수 계산
df["기존_위험도점수"] = 0
df["변경_위험도점수"] = 0

for col in base_weights.keys():
    df["기존_위험도점수"] += df[col] * base_weights[col]
    df["변경_위험도점수"] += df[col] * changed_weights[col]

# 순위 계산
df["기존_순위"] = df["기존_위험도점수"].rank(
    ascending=False,
    method="min"
).astype(int)

df["변경_순위"] = df["변경_위험도점수"].rank(
    ascending=False,
    method="min"
).astype(int)

# TOP5 비교
base_top5 = df.sort_values("기존_위험도점수", ascending=False).head(5)
changed_top5 = df.sort_values("변경_위험도점수", ascending=False).head(5)

base_top5_set = set(base_top5["자치구"])
changed_top5_set = set(changed_top5["자치구"])

same_top5 = base_top5_set & changed_top5_set
top5_retention_rate = len(same_top5) / 5 * 100

print("기존 가중치 TOP5")
print(base_top5[["자치구", "기존_위험도점수", "기존_순위"]])

print("\n변경 가중치 TOP5")
print(changed_top5[["자치구", "변경_위험도점수", "변경_순위"]])

print("\n공통 TOP5 지역:", same_top5)
print(f"TOP5 유지율: {top5_retention_rate:.1f}%")

기존 가중치 TOP5
   자치구  기존_위험도점수  기존_순위
1  강서구  0.422651      1
0  강북구  0.422352      2
3  도봉구  0.395420      3
2  노원구  0.388181      4
4  중랑구  0.385427      5

변경 가중치 TOP5
   자치구  변경_위험도점수  변경_순위
1  강서구  0.454427      1
0  강북구  0.444204      2
2  노원구  0.422251      3
4  중랑구  0.420217      4
3  도봉구  0.404649      5

공통 TOP5 지역: {'도봉구', '강서구', '중랑구', '노원구', '강북구'}
TOP5 유지율: 100.0%
